# AkasicDB Hybrid Search: Graph + Full-Text + Vector

In this notebook, you load synthetic product data into AkasicDB and run a single SQL query that combines **graph traversal (Cypher)**, **full-text search (BM25, texton)**, **vector similarity (vectoron)**, and **SQL filters**. Finally, you check the execution plan with PostgreSQL `EXPLAIN`.

## Prerequisites

* An AkasicDB instance
    * Create a Playground instance at [db.akasic.cloud](https://db.akasic.cloud), or
    * Install AkasicDB locally by following the [installation guide](https://docs.graphai.io/akasicdb-docs/latest/en/installation/installation/).
* A connection string
    * Google Colab: click the key icon in the left sidebar and save it as a secret named `DB_DSN`.
    * Other environments: set the `DATABASE_URL` environment variable.
* (Optional) A GPU runtime. In Colab, choose **Runtime → Change runtime type → T4 GPU**. Without a GPU, the notebook uses a smaller dataset and runs on the CPU.

## Step 1: Install dependencies

In [ ]:
%pip install --quiet akasicdb "psycopg[binary]" certifi "sentence-transformers>=2.7" "transformers>=4.51" accelerate

## Step 2: Connect to the database and enable the extension

The notebook uses the `DATABASE_URL` environment variable if it is set. Otherwise, it reads `DB_DSN` from Colab Secrets.

Enable the `akasicdb` extension once per database. `CASCADE` also installs the `vectoron` and `graphon` extensions that AkasicDB depends on. Running it again on a database where the extension is already enabled is safe.

Playground connection strings use `sslmode=verify-full&sslrootcert=system`, which verifies the server certificate. The OpenSSL bundled with `psycopg[binary]` may not find the operating system's certificate store, so the notebook points `SSL_CERT_FILE` to the `certifi` CA bundle.

`connect()` opens a connection and applies the session setup (vector type registration and `search_path`). If the connection drops in a later step, run `conn = connect()` and run that cell again.

Creating the extension and running `akasicdb_admin.initialize()` require administrator privileges. On a database that an administrator already initialized, such as a Playground instance, the notebook skips initialization if you don't have permission.

In [ ]:
import os

import certifi
import psycopg
from akasicdb.psycopg import register_vector

# The OpenSSL bundled with psycopg[binary] may not find the operating system's certificate store.
# Point it to the certifi CA bundle so that sslmode=verify-full&sslrootcert=system connections
# (the Playground default) can verify the server certificate. An existing SSL_CERT_FILE is kept.
os.environ.setdefault("SSL_CERT_FILE", certifi.where())


MSG_NEED_ADMIN = "You don't have permission to create the akasicdb extension. Ask your database administrator to run CREATE EXTENSION akasicdb CASCADE."
MSG_SKIP_INIT = "Skipping akasicdb_admin.initialize() because you don't have permission to run it. If an administrator already initialized this database, you can continue."


def get_dsn() -> str:
    dsn = os.getenv("DATABASE_URL")
    if dsn:
        return dsn
    try:
        from google.colab import userdata
    except ImportError:
        raise RuntimeError("Set the DATABASE_URL environment variable or save DB_DSN in Colab Secrets.")
    return userdata.get("DB_DSN")


def configure(conn: psycopg.Connection) -> None:
    """Session setup: register the vector type and put the example schema first in search_path."""
    register_vector(conn)
    conn.execute("SELECT set_config('search_path', 'hybrid_search_demo, ' || current_setting('search_path'), false)")


def connect() -> psycopg.Connection:
    """Open a new connection and apply the session setup. Also used to reconnect after a dropped connection."""
    conn = psycopg.connect(get_dsn(), autocommit=True)
    configure(conn)
    return conn


conn = psycopg.connect(get_dsn(), autocommit=True)
try:
    conn.execute("CREATE EXTENSION IF NOT EXISTS akasicdb CASCADE")
except psycopg.errors.InsufficientPrivilege:
    raise RuntimeError(MSG_NEED_ADMIN) from None
try:
    conn.execute("SELECT akasicdb_admin.initialize()")
except psycopg.errors.InsufficientPrivilege:
    print(MSG_SKIP_INIT)
configure(conn)

print(conn.execute("SELECT version()").fetchone()[0])

## Step 3: Create the schema

The example creates its tables in a dedicated schema, `hybrid_search_demo`, and its graph under the name `hybrid_search_demo_graph`, so that it doesn't touch your existing data. To make the notebook re-runnable, it drops and re-creates this schema and graph every time. Product description embeddings are stored as `vectoron.vector(1024)` to match the output dimension of Qwen3-Embedding-0.6B.

In [ ]:
for stmt in (
    "SELECT akasicdb.delete_graph('hybrid_search_demo_graph', true)",
    "SELECT akasicdb.undefine_graph('hybrid_search_demo_graph')",
):
    try:
        conn.execute(stmt)
    except psycopg.Error:
        pass  # Skip if the graph does not exist.

conn.execute("DROP SCHEMA IF EXISTS hybrid_search_demo CASCADE")
conn.execute("CREATE SCHEMA hybrid_search_demo")

conn.execute("""
CREATE TABLE brands (
  b_id  integer PRIMARY KEY,
  name  text    NOT NULL,
  hot   boolean NOT NULL DEFAULT false
)""")

conn.execute("""
CREATE TABLE products (
  id       integer PRIMARY KEY,
  b_id     integer NOT NULL REFERENCES brands(b_id),
  title    text    NOT NULL,
  category text    NOT NULL,
  price    integer NOT NULL,
  descr    text    NOT NULL,
  emb      vectoron.vector(1024)
)""")
print("ok")

## Step 4: Generate synthetic data

With a GPU, the notebook creates 10,000 products. Without one, it creates 2,000. Ten of the 50 brands are popular (`hot`), and their products are spread evenly across all categories.

Each product description combines a season, an item, and a feature.

In [ ]:
import random

import torch

if torch.cuda.is_available():
    DEVICE, DTYPE, N_PRODUCTS = "cuda", torch.float16, 10_000
else:
    DEVICE, DTYPE, N_PRODUCTS = "cpu", torch.float32, 2_000
print(f"Device: {DEVICE}, products: {N_PRODUCTS:,}")

random.seed(42)
N_BRANDS = 50

seasons = ["summer", "winter", "spring", "fall"]
items = ["running shoes", "sneakers", "sandals", "trekking boots", "slip-ons"]
feats = ["breathable", "lightweight", "well-cushioned", "waterproof"]
cats = ["shoes", "apparel", "bag", "acc", "outdoor"]

templates = [
    lambda s, i, f: f"Looking for {s} {i}? These {f} picks are a great choice.",
    lambda s, i, f: f"{f.capitalize()} {i} made for the {s} season.",
    lambda s, i, f: f"Stay comfortable through the {s} in these {f} {i}.",
]

brands = [(b, f"brand_{b}", b % 5 == 0) for b in range(1, N_BRANDS + 1)]

products = []
for g in range(1, N_PRODUCTS + 1):
    season = seasons[(g // 7) % len(seasons)]
    item = items[(g // 3) % len(items)]
    feat = feats[g % len(feats)]
    descr = templates[(g // 11) % len(templates)](season, item, feat)
    products.append((
        g,                                   # id
        (g // 5) % N_BRANDS + 1,             # b_id: assigned independently of the category
        f"product_{g}",                      # title
        cats[g % len(cats)],                 # category
        10_000 + random.randrange(190_000),  # price
        descr,
    ))

hot_brands = {b for b, _, hot in brands if hot}
hot_shoes = sum(1 for p in products if p[1] in hot_brands and p[3] == "shoes")
print(f"Shoes from popular brands: {hot_shoes}")
for p in products[:3]:
    print(p)

## Step 5: Generate embeddings

Qwen3-Embedding-0.6B is a multilingual embedding model. Encode product descriptions without an instruction, and encode search queries with `prompt_name="query"`. With `normalize_embeddings=True`, ordering by L2 distance (`<->`) is the same as ordering by cosine similarity.

Encoding 10,000 products takes a few minutes on a T4 GPU. Encoding 2,000 products on a CPU takes several minutes or more.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device=DEVICE,
    model_kwargs={"torch_dtype": DTYPE},
    tokenizer_kwargs={"padding_side": "left"},
)

embs = model.encode(
    [p[5] for p in products],
    batch_size=64 if DEVICE == "cuda" else 16,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(embs.shape)

## Step 6: Load the data

Load the products in batches of 1,000. Each batch runs in its own transaction. If the connection drops during loading, the notebook reconnects with `connect()` and retries the same batch, up to three times. The new session also gets `search_path`, so the later steps work as before. With `ON CONFLICT DO NOTHING`, loading the same batch twice doesn't create duplicates.

In [ ]:
import time

MAX_ATTEMPTS = 3


def insert_rows(sql: str, rows: list) -> None:
    """Insert a batch in one transaction. Reconnect and retry the same batch if the connection drops."""
    global conn
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            with conn.transaction(), conn.cursor() as cur:
                cur.executemany(sql, rows)
            return
        except psycopg.OperationalError as e:
            if attempt == MAX_ATTEMPTS:
                raise
            reason = str(e).strip().splitlines()[0]
            print(f"Connection lost. Reconnecting ({attempt}/{MAX_ATTEMPTS}): {reason}")
            time.sleep(2 * attempt)
            conn = connect()


insert_rows("INSERT INTO brands VALUES (%s, %s, %s) ON CONFLICT (b_id) DO NOTHING", brands)

rows = [(*p, emb) for p, emb in zip(products, embs)]
BATCH = 1000
for i in range(0, len(rows), BATCH):
    insert_rows(
        "INSERT INTO products (id, b_id, title, category, price, descr, emb) "
        "VALUES (%s, %s, %s, %s, %s, %s, %s) ON CONFLICT (id) DO NOTHING",
        rows[i:i + BATCH],
    )
    print(f"{min(i + BATCH, len(rows)):,} / {len(rows):,}")

conn.execute("ANALYZE brands, products")
loaded = conn.execute("SELECT count(*) FROM products").fetchone()[0]
print(f"Loaded products: {loaded:,}")

## Step 7: Create indexes

Create three kinds of indexes.

* **vectoron**: approximate nearest neighbor (ANN) search on the embeddings
* **texton**: full-text search (BM25) on the product descriptions. This is a whole-row index on `products.*`, and its `mapping` indexes the `descr` field with the language-neutral `standard` analyzer. The `@@` search operator uses this index.
* **B-tree**: the category filter

In [ ]:
conn.execute("CREATE INDEX products_emb_idx ON products USING vectoron (emb vector_l2_ops)")

conn.execute("""
CREATE INDEX products_texton_idx
  ON products
  USING texton ((products.*))
  WITH (mapping='{"fields":{"descr":{"type":"text"}}}')""")

conn.execute("CREATE INDEX products_cat_idx ON products (category)")
print("ok")

## Step 8: Define the graph

Define graph vertices and edges from the existing `products` and `brands` tables. The data is not loaded again.

* Vertices: `v_product` (products) and `v_brand` (brands)
* Edge: `e_made_by` (product → brand)

After creating the graph, the notebook collects statistics on the graph tables (`ANALYZE`). The planner needs these statistics to use the edge index, as you see in Step 10.

The Community Edition allows one graph per database. If another graph already exists, the notebook stops and shows how to delete the existing graph.

In [ ]:
MSG_GRAPH_LIMIT = (
    "The Community Edition allows only one graph, and this database already has another graph. "
    "Check it with SELECT * FROM akasicdb.list_graph(). If you don't need it, "
    "delete it with SELECT akasicdb.delete_graph('<graph name>', true) and run this cell again."
)

try:
    conn.execute("SELECT akasicdb.define_graph('hybrid_search_demo_graph')")
except psycopg.Error as e:
    if "community edition allows" in str(e):
        raise RuntimeError(MSG_GRAPH_LIMIT) from None
    raise

conn.execute("""
SELECT akasicdb.define_vertex('hybrid_search_demo_graph', 'v_product',
  ARRAY['product_id integer', 'title text'],
  'hybrid_search_demo.products',
  'SELECT id AS product_id, title FROM hybrid_search_demo.products')""")

conn.execute("""
SELECT akasicdb.define_vertex('hybrid_search_demo_graph', 'v_brand',
  ARRAY['b_id integer', 'name text', 'hot boolean'],
  'hybrid_search_demo.brands',
  'SELECT b_id, name, hot FROM hybrid_search_demo.brands')""")

conn.execute("""
SELECT akasicdb.define_edge('hybrid_search_demo_graph', 'e_made_by',
  'v_product', 'v_brand',
  null,
  'SELECT null FROM hybrid_search_demo.products p, hybrid_search_demo.brands b WHERE p.b_id = b.b_id',
  'hybrid_search_demo.products p', 'hybrid_search_demo.brands b')""")

conn.execute("SELECT akasicdb.create_graph('hybrid_search_demo_graph')")

# Collect statistics on the new graph tables so that the planner can use the edge index.
graph_tables = conn.execute(
    "SELECT format('%I.%I', schemaname, tablename) FROM pg_tables "
    "WHERE schemaname = 'hybrid_search_demo_graph'"
).fetchall()
for (table,) in graph_tables:
    conn.execute(f"ANALYZE {table}")
print("graph ready")

## Step 9: Run a hybrid query

A single SQL query combines four conditions.

1. **Graph (Cypher)**: products made by popular brands (`hot = true`)
2. **SQL filter**: `category = 'shoes'`
3. **Full-text search (BM25)**: descriptions that contain both "summer" and "running". `akasicdb.text_score(p)` shows the BM25 score.
4. **Vector similarity**: ordered by semantic similarity to "breathable running shoes for summer". `dist` in the results is the L2 distance to the query vector.

Because the data is synthetic, many products have the same BM25 score. In that case, the vector distance (`dist`) decides the order.

In [ ]:
q_emb = model.encode(
    ["breathable running shoes for summer"],
    prompt_name="query",
    normalize_embeddings=True,
)[0]

HYBRID_SQL = """
SELECT p.id, p.title, p.price, akasicdb.text_score(p) AS bm25,
       p.emb <-> %s::vectoron.vector AS dist, left(p.descr, 40) AS descr
FROM   products p
JOIN   akasicdb.cypher('hybrid_search_demo_graph', $$
           MATCH (pr:v_product)-[:e_made_by]->(b:v_brand)
           WHERE b.hot = true
           RETURN pr.product_id AS product_id
       $$) AS g (product_id integer)
       ON g.product_id = p.id                          -- graph (Cypher)
WHERE  p.category = 'shoes'                            -- SQL filter
  AND  p @@ akasicdb.text_query('summer AND running')  -- full-text search (BM25)
ORDER  BY dist                                          -- vector similarity (L2)
LIMIT  10
"""

rows = conn.execute(HYBRID_SQL, (q_emb,)).fetchall()
for id_, title, price, bm25, dist, descr in rows:
    print(f"{id_:>5}  {title:<13} price={price:>7,}  bm25={bm25:.3f}  dist={dist:.4f}  {descr}")
print(f"Hybrid results: {len(rows)}")

## Step 10: Check the execution plan

Use standard PostgreSQL `EXPLAIN` to see the plan for the hybrid query.

In [ ]:
import re

plan = conn.execute("EXPLAIN (ANALYZE, BUFFERS) " + HYBRID_SQL, (q_emb,)).fetchall()
for (line,) in plan:
    # Shorten the 1024-dimensional query vector literal to '[...]'.
    print(re.sub(r"'\[[^']*\]'", "'[...]'", line))

### How to read the plan

1. **Full-text search**: `Index Scan using products_texton_idx on products p` means the BM25 search ran on the texton index. A texton `@@` condition is always evaluated through this index scan, and the `category` condition is applied as a `Filter` on the same node.
2. **Vector ordering**: A `Sort` node on the `p.emb <-> ...` distance means the planner computed exact distances only for the candidates that passed the earlier conditions. The planner chooses this path when full-text and graph conditions narrow the candidates a lot.
3. **Graph**: The Cypher query expands into joins over the graph topology tables (`v_product_topo`, `v_brand_topo`, `e_made_by_topo`), so it is optimized together with the other conditions in a single plan. Edges from popular brands to their products are found with `Index Only Scan using e_made_by_topo_dst_vid_idx` (the edge index). The planner chooses this index because Step 8 collected statistics on the graph tables. Without statistics, the plan may read the whole edge table with `Seq Scan on e_made_by_topo` instead.

Try changing the conditions (`WHERE b.hot`, `category`, the full-text query) and compare how the plan changes.